In [43]:
from minsearch import Index

import minsearch
import json

with open('parse/documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

documents = []
for dict_course in docs_raw:
    course = dict_course['course']
    for dict_document in dict_course['documents']:
        documents.append({ 
        'course': course,
        'text': dict_document['text'],
        'section': dict_document['section'],
        'question': dict_document['question']
        })
    
index = Index(
    text_fields=["question","text","section"],
    keyword_fields=["course"]
)
index.fit(documents)

In [44]:
def search(query):
    filter_dict = {"course": "data-engineering-zoomcamp"}
    results = index.search(
        query=query, 
        boost_dict={'question': 3.0, 'section':0.5},
        filter_dict=filter_dict,
        num_results=3
    )
    return results

In [45]:
def build_prompt(query, search_results):
    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"

    prompt_template = f"""
        You are a course teaching assistant. Answer the QUESTION based on the CONTEXT. 
        Use only the facts from the CONTEXT when answering the QUESTION.
        If the CONTEXT doesn't contain the answer output NONE.
        
        QUESTION: {query}
        
        CONTEXT: 
        {context}""".strip()

    prompt = prompt_template.format(query=query, context=context).strip()
    return prompt

In [46]:
query = 'how do i run kafka?'
search_results = search(query)

In [47]:
print(search_results)

[{'course': 'data-engineering-zoomcamp', 'text': "Solution from Alexey: create a virtual environment and run requirements.txt and the python files in that environment.\nTo create a virtual env and install packages (run only once)\npython -m venv env\nsource env/bin/activate\npip install -r ../requirements.txt\nTo activate it (you'll need to run it every time you need the virtual env):\nsource env/bin/activate\nTo deactivate it:\ndeactivate\nThis works on MacOS, Linux and Windows - but for Windows the path is slightly different (it's env/Scripts/activate)\nAlso the virtual environment should be created only to run the python file. Docker images should first all be up and running.", 'section': 'Module 6: streaming with kafka', 'question': 'Module “kafka” not found when trying to run producer.py'}, {'course': 'data-engineering-zoomcamp', 'text': 'In the project directory, run:\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java', 'section': 'Mo

In [48]:
prompt = build_prompt(query, search_results)

In [59]:
def llm(prompt):
    from openai import OpenAI
    client = OpenAI(api_key="")
    response = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {"role":"user", "content":query }
        ]
    )
    return response.choices[0].message.content

In [62]:
def rag(query='how do i run kafka?'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [63]:
print(rag())

Running Apache Kafka involves several steps, including setting up the environment, downloading and starting Kafka, and producing and consuming messages. Here's a basic guide to get you started:

### Prerequisites

1. **Java Development Kit (JDK):** Kafka requires a Java runtime environment. Ensure you have JDK 8 or later installed. Verify your installation by running `java -version` in your terminal or command prompt.

2. **Zookeeper:** Kafka uses Zookeeper to manage distributed brokers. Since Apache Kafka 2.8, Zookeeper is no longer required for local setups, but it's still a dependency when setting a Kafka cluster.

### Steps to Run Kafka

#### 1. Download Kafka
- Go to the [Apache Kafka downloads page](https://kafka.apache.org/downloads).
- Download the latest binary release for your operating system.
- Extract the downloaded files to a convenient location on your system.

#### 2. Start Zookeeper
Though newer versions of Kafka can run without Zookeeper in a local setup, here is how 